# Lecture 7 — The Krusell-Smith Method

**Computational Methods for Heterogeneous-Agent Macro**

Jeffrey Sun


## 0 · Setup

We import `HouseholdStages` plus the usual numerical helpers.


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Statistics
using LinearAlgebra: I
using Random: MersenneTwister
using Plots


## 1 · The state explosion

In L04 we solved Aiyagari at one steady state. In L05 we did a *deterministic* transition between two steady states. In both, the path of aggregate $K$ was known *with certainty* once we conjectured it; the inner household solve never had to integrate over future $K$.

**Now** add an aggregate productivity shock $Z_t$ — TFP follows a Markov chain over $\{Z_{\text{bad}}, Z_{\text{good}}\}$. Prices $r_t, w_t$ depend on $(Z_t, K_t)$. Crucially, **the household cannot anticipate $K_{t+1}$ from $K_t$ alone** — $K_{t+1}$ is the integral of *all* households' savings choices, which depend on the entire wealth distribution $\Lambda_t$.

So the agent's own state space has to include $\Lambda_t$ to make rational decisions:
$$V(b_i, z_i; \Lambda, Z) = \max_{c_i, b_i'} u(c_i) + \beta\, \mathbb{E}[V(b_i', z_i'; \Lambda', Z') \mid z_i, Z].$$

But $\Lambda_t$ is infinite-dimensional. Tabulating $V$ over it is hopeless. This is the **Krusell-Smith problem**.


## 2 · The K-S workaround

Krusell and Smith (1998) make a **bounded-rationality assumption**: agents do not track the entire distribution $\Lambda_t$. They track only its first moment — aggregate capital $K_t$ — and forecast next period's $K$ from $(K_t, Z_t)$ via a **parametric Approximate Law of Motion** (ALM):
$$\log K_{t+1} = a_0(Z_t) + a_1(Z_t) \log K_t.$$

Given $(a_0, a_1)$, the household's state collapses to $(b, y, Z, K)$ — finite-dimensional, solvable by VFI. **But** agents' beliefs about $K_{t+1}$ may not match the true equilibrium dynamics. The K-S iteration repairs this:

1. **Pick** an initial ALM $(a_0^{(0)}, a_1^{(0)})$.
2. **Solve** the household problem at the current ALM: get $V^*$ and policy $b'^*(b, y, Z, K)$.
3. **Simulate** a long economy with realised $\{Z_t\}$ and *true* aggregation $K_t = \int b\, \mathrm{d}\Lambda_t$.
4. **Refit** $(a_0, a_1)$ by OLS of $\log K_{t+1}$ on $\log K_t$, separately per $Z$ state.
5. **Iterate** until $(a_0, a_1)$ stops moving.


## 3 · Parameters

Canonical K-S calibration: log utility, $\beta = 0.96$, $\alpha = 0.36$, $\delta = 0.025$. Aggregate $Z \in \{0.99, 1.01\}$ with high persistence. Idiosyncratic $z \in \{0.07, 1.0\}$ (unemployed/employed).

Grids are deliberately small for pedagogical speed — $N_w = 80$, $N_K = 5$.


In [ ]:
@kwdef struct KSParams
    β::Float64 = 0.96
    γ::Float64 = 1.0
    α::Float64 = 0.36
    δ::Float64 = 0.025
    z_grid::Vector{Float64} = [0.07, 1.0]
    P_z::Matrix{Float64}    = [0.6   0.4;
                               0.05  0.95]
    Z_vals::Vector{Float64} = [0.99, 1.01]
    P_Z::Matrix{Float64}    = [0.875 0.125;
                               0.125 0.875]
    N_w::Int       = 80
    w_min::Float64 = 0.0
    w_max::Float64 = 80.0
    N_K::Int       = 5
    K_min::Float64 = 10.5
    K_max::Float64 = 14.5
end
const ks_params = KSParams()
p = ks_params
@printf "β = %.3f, γ = %.2f, α = %.2f, δ = %.3f\n" p.β p.γ p.α p.δ

## 4 · Prices and effective labour

Cobb-Douglas production with TFP $Z$ and effective labour $L$:
$$r(Z, K) = \alpha\, Z\, (K/L)^{\alpha-1} - \delta, \qquad w(Z, K) = (1-\alpha)\, Z\, (K/L)^\alpha.$$

Effective labour is the stationary-distribution average of $z$ — a constant in this model since $P_z$ is independent of $Z$.


In [ ]:
"""CLAUDE
Stationary-weighted average of `z_grid` under Markov chain `P_z`.
"""
function ks_effective_labor(P_z::AbstractMatrix, z_grid::AbstractVector)
    n = size(P_z, 1)
    A = P_z' - I(n)
    A[end, :] .= 1.0
    rhs = zeros(n); rhs[end] = 1.0
    π = A \ rhs
    return sum(z_grid .* π)
end

function ks_prices(K::Real, Z::Real, p::KSParams)
    L = ks_effective_labor(p.P_z, p.z_grid)
    r = p.α * Z * (K/L)^(p.α - 1) - p.δ
    w = (1 - p.α) * Z * (K/L)^p.α
    return (; r, w)
end

L_eff = ks_effective_labor(p.P_z, p.z_grid)
@printf "L_eff = %.4f\n" L_eff


## 5 · Aiyagari deterministic SS at $Z = 1$

Before adding aggregate uncertainty, find the deterministic steady state $\bar K$ at $Z = 1$ using L04's tatonnement on $K$. This is the natural "center" for the K-grid in §6 and the initial $\Lambda_0$ for the K-S simulation.


In [ ]:
_u_crra(c, ::Union{Val{1}, Val{1.0}}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

"""CLAUDE
Build a 2D (wealth, z) household chain whose income stage reads
r and w directly from env. Used for the Z = 1 Aiyagari tatonnement and
later as a forward-stepper at realised (Z_t, K_t) during simulation.
"""
function ks_household_2d(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z, p.z_grid),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    return define_moments!(z_shock ∘ income ∘ savings; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

"""CLAUDE
Damped tatonnement on K at Z = 1, using `ks_household_2d`. Returns
the converged K plus the 2D distribution Λ_2d, which serves as the
K-grid center and the K-S simulation's initial condition. A loose
rtol of 1e-2 suffices since K_bar is only a grid center.
"""
function aiyagari_steady_state_at_Z(p::KSParams; Z::Float64=1.0, verbose::Bool=true)
    hh = ks_household_2d(p)
    K = 12.0
    V, Λ = nothing, nothing
    for iter in 1:200
        env = make_env(hh; ks_prices(K, Z, p)...)
        kw = (; lambda_tol=1e-5, lambda_maxiter=50_000)
        res = isnothing(V) ? solve_steady_state_given_env!(hh, env; kw...) :
              solve_steady_state_given_env!(hh, env; V_init=V, Λ_init=Λ, kw...)
        V, Λ = res.V, res.Λ
        K_sup = res.moments.K_supplied
        K_err = abs(K_sup - K) / K
        verbose && @printf "  iter %d: K = %.3f → K_supplied = %.3f, err = %.5f\n" iter K K_sup K_err
        K_err <= 1e-2 && return (; K, Λ, V, iters=iter)
        K = 0.95 * K + 0.05 * K_sup
    end
    error("aiyagari_steady_state_at_Z: did not converge")
end

println("Finding deterministic Aiyagari SS at Z = 1.0...")
@time det_ss = aiyagari_steady_state_at_Z(p; verbose=false)
@printf "K̄ = %.4f in %d iters\n" det_ss.K det_ss.iters
K_bar = det_ss.K
Λ_0   = det_ss.Λ

## 6 · The household block as five stages

Time order within a period:

1. **`z_shock`** — idiosyncratic productivity $z_i^t \to z_i^{t+1}$ (Markov on `:z`).
2. **`income`** — wealth update with $r, w$ at the cell's current $(Z, K)$.
3. **`savings`** — choose $b_i^{t+1}$ on the wealth grid.
4. **`K_evolve`** — deterministic $K_t \to K_{t+1}$ under the agent's ALM. Linear interpolation across the K-grid.
5. **`Z_shock`** — $Z_t \to Z_{t+1}$ (Markov).

`K_evolve` and `Z_shock` sit at the *end* of the chain so the price-reading stage (2) sees the start-of-period $(Z, K)$ via cell coordinates.


In [ ]:
function ks_household(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z,      p.z_grid),
        StateAxis(:Z_idx,  [1, 2]),
        StateAxis(:K,      continuous_grid(p.K_min, p.K_max; length=p.N_K)),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> begin
        Z = p.Z_vals[cell.Z_idx]
        pr = ks_prices(cell.K, Z, p)
        (1 + pr.r) * cell.wealth + pr.w * cell.z
    end)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    K_evolve = WealthChangeStage(layout; wealth_axis=:K, wealth_post=(cell; env) -> begin
        a0 = env.a0[cell.Z_idx]
        a1 = env.a1[cell.Z_idx]
        exp(a0 + a1 * log(cell.K))
    end)
    Z_shock = MarkovStage(layout; axis=:Z_idx, transition=p.P_Z)
    chain = z_shock ∘ income ∘ savings ∘ K_evolve ∘ Z_shock
    return define_moments!(chain; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

hh = ks_household(p)
N_w, N_z, N_Z, N_K = layout_size(first(hh.spec.stages).input_layout)
@printf "Layout: %d × %d × %d × %d = %d cells\n" N_w N_z N_Z N_K (N_w*N_z*N_Z*N_K)

## 7 · A first VFI given a candidate ALM

Initial guess: a constant ALM — the agent believes $K_{t+1} = \bar K$ regardless of $(Z, K)$. That is, $a_0(Z) = \log \bar K$ and $a_1(Z) = 0$ for both $Z$ states. Solve the 4D VFI under this rule.


In [ ]:
a0_init = [log(K_bar), log(K_bar)]
a1_init = [0.0, 0.0]
env0 = make_env(hh; a0=a0_init, a1=a1_init)
println("Solving 4D household VFI at the initial ALM...")
@time ss = solve_steady_state_given_env!(hh, env0; lambda_tol=1e-5, lambda_maxiter=50_000)
@printf "VFI iters: %d, Λ iters: %d, ΣΛ = %.10f\n" ss.history.vfi_iters ss.history.lambda_iters sum(ss.Λ)

## 8 · Simulation under realised $(Z_t, K_t)$

We maintain $\Lambda_t$ as a **2D** distribution on $(b, z)$, and track $(Z_t, K_t)$ as realised scalars. Each period:

1. Compute $K_t = \sum_{b,z} b \cdot \Lambda_t[b, z]$.
2. Lift $\Lambda_t$ to a 4D distribution concentrated at $(Z_t, K_t)$ — share-redistributed between the two adjacent K-grid points.
3. Run **only the first three stages** of the chain (`z_shock ∘ income ∘ savings`) on the 4D distribution. This avoids the chain's `K_evolve` and `Z_shock`, which would impose the agent's *belief* about K/Z dynamics rather than the *realised* dynamics.
4. Marginalise the 4D result over $(Z, K)$ to recover $\Lambda_{t+1}$ on $(b, z)$.
5. Draw $Z_{t+1} \mid Z_t$ from the Markov chain.

Step 3 is where the chain machinery still pays — `income`'s share-based wealth redistribution and `savings`' policy lookup are non-trivial to re-implement by hand.


In [ ]:
"""CLAUDE
Build a 4D chain with only the first three stages (z_shock, income,
savings). Used to push Λ forward at realised (Z_t, K_t) without the
agent-belief K/Z dynamics in K_evolve/Z_shock.
"""
function ks_household_sim(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z,      p.z_grid),
        StateAxis(:Z_idx,  [1, 2]),
        StateAxis(:K,      continuous_grid(p.K_min, p.K_max; length=p.N_K)),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> begin
        Z = p.Z_vals[cell.Z_idx]
        pr = ks_prices(cell.K, Z, p)
        (1 + pr.r) * cell.wealth + pr.w * cell.z
    end)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    return z_shock ∘ income ∘ savings
end

"""CLAUDE
Linear-interp weights between two K-grid points bracketing K_t.
Returns (i_lo, i_hi, w_lo) where mass share w_lo goes to grid point
i_lo and 1-w_lo to i_hi.
"""
function _K_neighbours(K_grid::AbstractVector, K_t::Real)
    K_t_c = clamp(K_t, K_grid[1], K_grid[end])
    i_hi = findfirst(>=(K_t_c), K_grid)::Int
    if i_hi == 1
        return (1, 1, 1.0)
    end
    i_lo = i_hi - 1
    w_lo = (K_grid[i_hi] - K_t_c) / (K_grid[i_hi] - K_grid[i_lo])
    return (i_lo, i_hi, w_lo)
end

"""CLAUDE
Push Λ_2d forward one period using the 4D simulation chain `hh_sim`,
which must have its kernels seated (call `backward!` first to populate
the savings policy from the converged V*). Returns Λ_2d_{t+1} and K_t.
"""
function ks_simulate_step(hh_sim::ChainStage, p::KSParams, Λ_2d::AbstractMatrix, Z_t_idx::Int, K_t::Real)
    K_grid = axisvalues(first(hh_sim.spec.stages).input_layout.axes[4])
    i_lo, i_hi, w_lo = _K_neighbours(K_grid, K_t)
    N_w, N_z = size(Λ_2d)
    N_Z = length(p.Z_vals)
    N_K = length(K_grid)
    # Lift to 4D, share-redistributing K_t between i_lo and i_hi.
    Λ_4d = zeros(N_w, N_z, N_Z, N_K)
    for z in 1:N_z, b in 1:N_w
        m = Λ_2d[b, z]
        iszero(m) && continue
        Λ_4d[b, z, Z_t_idx, i_lo] += m * w_lo
        if i_lo != i_hi
            Λ_4d[b, z, Z_t_idx, i_hi] += m * (1 - w_lo)
        end
    end
    Λ_4d_next = forward!(hh_sim, Λ_4d)
    # Marginalise (Z, K) back to (b, z).
    Λ_2d_next = dropdims(sum(Λ_4d_next; dims=(3, 4)); dims=(3, 4))
    return Λ_2d_next
end

"""CLAUDE
Simulate T periods. Caller must have called `backward!(hh_sim, V_end,
env)` for some env in advance so the savings policy is seated. Returns
(K_path, Z_idx_path) of length T.
"""
function ks_simulate(hh_sim::ChainStage, p::KSParams, T::Int; Λ_init::AbstractMatrix, Z_init_idx::Int=1)
    rng = MersenneTwister(42)
    wgrid = axisvalues(first(hh_sim.spec.stages).input_layout.axes[1])
    Λ = copy(Λ_init)
    Z_idx = Z_init_idx
    K_path     = zeros(T)
    Z_idx_path = zeros(Int, T)
    for t in 1:T
        K_t = sum(wgrid .* sum(Λ; dims=2))
        K_path[t]     = K_t
        Z_idx_path[t] = Z_idx
        Λ = ks_simulate_step(hh_sim, p, Λ, Z_idx, K_t)
        u = rand(rng)
        Z_idx = u < p.P_Z[Z_idx, 1] ? 1 : 2
    end
    return (; K_path, Z_idx_path)
end

**Seat the simulation chain.** Before `ks_simulate_step` can use the savings policy, we need to populate the chain's savings kernel via `backward!`. We solve the sim chain's backward at a representative env using the converged V from §7 as terminal value — this materialises the savings policy under the converged household perception.


In [ ]:
hh_sim = ks_household_sim(p)
# Backward fills the savings kernel.policy. The receipt/savings stages don't
# read env directly except for closures that use cell coords (Z_idx, K) — but
# the closure-bound stages take prices from the cell's (Z, K), not from env.
# So an empty env suffices.
backward!(hh_sim, copy(ss.V), (;))
@printf "ks_household_sim backward done; policy shape = %s\n" string(size(hh_sim.buffer.stages[3].kernel.policy))


## 9 · A first simulation under the constant ALM

Use the converged 4D policy and the deterministic-SS $\Lambda_0$. Simulate $T = 2000$ periods, burn-in 500.


In [ ]:
T_sim  = 2000
T_burn = 500
println("Simulating $T_sim periods under the initial (constant) ALM...")
@time sim0 = ks_simulate(hh_sim, p, T_sim; Λ_init=Λ_0, Z_init_idx=1)
@printf "K-path: min = %.3f, mean = %.3f, max = %.3f, std = %.3f\n" minimum(sim0.K_path) mean(sim0.K_path) maximum(sim0.K_path) std(sim0.K_path)
@printf "Fraction Z = good: %.3f\n" (sum(sim0.Z_idx_path .== 2) / length(sim0.Z_idx_path))

**Inspection plot.** $K_t$ moves with $Z_t$. Under the *constant* ALM the household believes $K_{t+1}$ is always $\bar K$, but in reality $K$ co-moves with $Z$.


In [ ]:
window = (T_burn + 1):(T_burn + 200)
plt_K = plot(window, sim0.K_path[window]; lw=2, label="K_t",
             xlabel="period t", ylabel="K_t",
             title="K-path under initial (constant) ALM",
             size=(700, 320))
hline!(plt_K, [K_bar]; color=:gray, ls=:dash, label="K̄ (initial ALM)")
plt_Z = plot(window, [p.Z_vals[sim0.Z_idx_path[t]] for t in window];
             lw=2, color=:firebrick, label="Z_t",
             xlabel="period t", ylabel="Z_t",
             title="Realised TFP path", size=(700, 160))
plot(plt_K, plt_Z; layout=(2, 1), size=(700, 480))

## 10 · Refitting the ALM

Regress $\log K_{t+1}$ on $\log K_t$ **separately by $Z_t$**. The OLS coefficients are the updated $(a_0(Z), a_1(Z))$.


In [ ]:
"""CLAUDE
OLS regression of log K_{t+1} on (1, log K_t), restricted to periods
with Z_t == z_idx and t in `window`. Returns (a0, a1, R², n).
"""
function regress_ALM(K_path::AbstractVector, Z_idx_path::AbstractVector, z_idx::Int, window::AbstractUnitRange)
    rows = [t for t in window if Z_idx_path[t] == z_idx]
    isempty(rows) && error("regress_ALM: no observations for z_idx = $z_idx")
    x = log.(K_path[rows])
    y = log.(K_path[rows .+ 1])
    x̄, ȳ = mean(x), mean(y)
    Sxx = sum((x .- x̄).^2)
    Sxy = sum((x .- x̄) .* (y .- ȳ))
    β̂  = Sxy / Sxx
    α̂  = ȳ - β̂ * x̄
    ŷ  = α̂ .+ β̂ .* x
    ss_res = sum((y .- ŷ).^2)
    ss_tot = sum((y .- ȳ).^2)
    R²     = 1 - ss_res / ss_tot
    return (; a0=α̂, a1=β̂, R², n=length(rows))
end

window_fit = (T_burn + 1):(T_sim - 1)
fit_bad  = regress_ALM(sim0.K_path, sim0.Z_idx_path, 1, window_fit)
fit_good = regress_ALM(sim0.K_path, sim0.Z_idx_path, 2, window_fit)
@printf "Z = bad  (Z=%.2f): a0 = %+.4f, a1 = %+.4f, R² = %.4f  (n = %d)\n" p.Z_vals[1] fit_bad.a0  fit_bad.a1  fit_bad.R²  fit_bad.n
@printf "Z = good (Z=%.2f): a0 = %+.4f, a1 = %+.4f, R² = %.4f  (n = %d)\n" p.Z_vals[2] fit_good.a0 fit_good.a1 fit_good.R² fit_good.n

## 11 · The K-S outer iteration

Wrap VFI (§7), simulation (§8–9), and refit (§10) in an outer loop. Stop when the ALM coefficients stop moving.


In [ ]:
function ks_iterate(p::KSParams, K_bar::Float64, Λ_0::AbstractMatrix; maxiter::Int=15)
    hh     = ks_household(p)
    hh_sim = ks_household_sim(p)
    T_sim  = 2000
    T_burn = 500

    a0 = [log(K_bar), log(K_bar)]
    a1 = [0.0, 0.0]
    R²_hist   = NTuple{2, Float64}[]
    sim_final = nothing

    for it in 1:maxiter
        env = make_env(hh; a0, a1)
        _ss = solve_steady_state_given_env!(hh, env; lambda_tol=1e-5, lambda_maxiter=50_000)
        # Re-seat the sim chain's policy from the updated V_ss.
        backward!(hh_sim, copy(_ss.V), (;))

        sim = ks_simulate(hh_sim, p, T_sim; Λ_init=Λ_0, Z_init_idx=1)
        window_fit = (T_burn + 1):(T_sim - 1)
        fit_bad  = regress_ALM(sim.K_path, sim.Z_idx_path, 1, window_fit)
        fit_good = regress_ALM(sim.K_path, sim.Z_idx_path, 2, window_fit)

        a0_new = [fit_bad.a0, fit_good.a0]
        a1_new = [fit_bad.a1, fit_good.a1]
        Δ = maximum(abs.(vcat(a0_new .- a0, a1_new .- a1)))
        @printf "  iter %2d: R² = (%.5f, %.5f), Δ = %.5f, K mean = %.3f\n" it fit_bad.R² fit_good.R² Δ mean(sim.K_path[window_fit])
        push!(R²_hist, (fit_bad.R², fit_good.R²))
        sim_final = sim

        a0 = 0.5 .* a0 .+ 0.5 .* a0_new
        a1 = 0.5 .* a1 .+ 0.5 .* a1_new

        Δ < 1e-3 && return (; a0, a1, R²_hist, sim=sim_final, iters=it)
    end
    error("ks_iterate: did not converge")
end

println("Running K-S outer iteration...")
@time res_ks = ks_iterate(p, K_bar, Λ_0; maxiter=18)
@printf "✓ Converged in %d outer iterations.\n" res_ks.iters

## 12 · The punchline — $R^2 > 0.996$

The converged ALM regression has $R^2$ effectively equal to one. The household-block dynamics implied by the converged policy are *almost perfectly* described by a one-line log-linear rule.

This is **approximate aggregation**: in this baseline calibration, the wealth distribution barely matters for the dynamics of $K$. Agents who only know $K_t$ and $Z_t$ can forecast $K_{t+1}$ as well as agents who knew the entire $\Lambda_t$.

Why? The key driver is Cobb-Douglas production with relatively patient agents and CRRA utility — household savings policies are approximately linear in wealth in the relevant range, so the *integral* (aggregate $K$) is determined by aggregate $K$ alone, not by higher moments. Krusell & Smith (1998) §3.2 discuss this explicitly.


In [ ]:
@printf "Converged ALM coefficients:\n"
@printf "  Z = bad  (Z=%.2f): log K_{t+1} = %+.4f + %+.4f log K_t\n" p.Z_vals[1] res_ks.a0[1] res_ks.a1[1]
@printf "  Z = good (Z=%.2f): log K_{t+1} = %+.4f + %+.4f log K_t\n" p.Z_vals[2] res_ks.a0[2] res_ks.a1[2]
final_R² = res_ks.R²_hist[end]
@printf "Final R²: (bad: %.5f, good: %.5f)\n" final_R²[1] final_R²[2]


In [ ]:
sim   = res_ks.sim
wfit  = (T_burn + 1):(T_sim - 1)
mask_bad  = sim.Z_idx_path[wfit] .== 1
mask_good = sim.Z_idx_path[wfit] .== 2
x_bad  = log.(sim.K_path[wfit][mask_bad])
y_bad  = log.(sim.K_path[wfit .+ 1][mask_bad])
x_good = log.(sim.K_path[wfit][mask_good])
y_good = log.(sim.K_path[wfit .+ 1][mask_good])

plt_scatter = plot(title="ALM fit at K-S convergence",
                   xlabel="log K_t", ylabel="log K_{t+1}",
                   size=(640, 420), legend=:topleft)
scatter!(plt_scatter, x_bad,  y_bad;  ms=2, alpha=0.3, label="Z = bad")
scatter!(plt_scatter, x_good, y_good; ms=2, alpha=0.3, label="Z = good", color=:firebrick)
x_grid = range(min(minimum(x_bad), minimum(x_good)),
               max(maximum(x_bad), maximum(x_good)); length=50)
plot!(plt_scatter, x_grid, res_ks.a0[1] .+ res_ks.a1[1] .* x_grid;
       lw=2, color=:steelblue, label="ALM (Z=bad)")
plot!(plt_scatter, x_grid, res_ks.a0[2] .+ res_ks.a1[2] .* x_grid;
       lw=2, color=:firebrick, label="ALM (Z=good)")

## 13 · The Den Haan critique

$R^2$ averages over many periods and can hide systematic bias. Den Haan (2010) proposes a sharper diagnostic: simulate $K$ forward over many periods using *only* the ALM (no agent re-aggregation) and compare to the true path.

If the ALM is a faithful summary, the two paths should track each other. In our baseline calibration they do — to within a few percent over hundreds of periods.


In [ ]:
function ks_alm_forward(K_0::Real, Z_idx_path::AbstractVector, a0, a1)
    T = length(Z_idx_path)
    K_alm = zeros(T)
    K_alm[1] = K_0
    for t in 1:(T - 1)
        z = Z_idx_path[t]
        K_alm[t+1] = exp(a0[z] + a1[z] * log(K_alm[t]))
    end
    return K_alm
end

K_alm_path = ks_alm_forward(sim.K_path[T_burn + 1], sim.Z_idx_path[T_burn + 1 : end],
                            res_ks.a0, res_ks.a1)
window_dh = 1:min(800, length(K_alm_path))
true_path = sim.K_path[T_burn + 1 : T_burn + length(K_alm_path)]
max_err_pct = 100 * maximum(abs.(K_alm_path[window_dh] .- true_path[window_dh]) ./ true_path[window_dh])
@printf "Den Haan diagnostic: max |%%error| over %d periods = %.2f%%\n" length(window_dh) max_err_pct

plot(window_dh, true_path[window_dh]; lw=2, label="True K_t (full aggregation)",
     xlabel="period (post-burn-in)", ylabel="K",
     title="Den Haan diagnostic", size=(720, 380))
plot!(window_dh, K_alm_path[window_dh]; lw=2, ls=:dash,
      label="ALM-only K_t (no aggregation)", color=:firebrick)

**Reading the plot.** The dashed (ALM-only) path stays within a few percent of the solid (true) path. In this calibration K-S's bounded-rationality assumption is essentially exact — the ALM is not just descriptively accurate, it is *prescriptively* good enough that agents using it make decisions indistinguishable from agents using the full distribution.

This is the headline-grabbing result that made K-S famous. **But** approximate aggregation is a property of *this* baseline calibration, not a general feature of HA models. Modify the calibration — non-CRRA preferences, occupation choice, fat-tailed income shocks, large idiosyncratic differences in MPC — and the linear ALM can break down. That's where parametric ALMs fail and **neural-network parameterisations** (next lecture) take over.


## 14 · Foreshadow L08

L07 gave you a K-S baseline that works because of approximate aggregation. L08 generalises:

- **Replace** the parametric ALM with $V_\theta(b_i, z_i, Z, K)$ — a neural network. No log-linear assumption.
- **Train** $V_\theta$ on the Bellman residual, sampling over both individual and aggregate states (the "Pooled Bellman" residual).
- **Drop** both the ALM and the regression. Agents' expectations live inside $V_\theta$.
- **Headline:** "Continuation Value is All You Need" — the same idea, parameterised non-parametrically.

CVIAYN inherits K-S's stage-based structure (the same `z_shock ∘ income ∘ savings ∘ K_evolve ∘ Z_shock` chain) but trains $V_\theta$ instead of iterating an ALM-plus-VFI loop.
